# Fabric Ontology Knowledge Source: Airline Ops

This notebook demonstrates the Fabric Ontology Knowledge Source path using the repo's Airline Ops sample contract.

Deployment mode mapping: use this notebook with `byo-fabric` when you already have a Fabric workspace and ontology, or with `full` after the greenfield deployment creates the sample Fabric assets.

It proves the minimum Fabric loop:

1. Inspect the sample Airline Ops data and expected facts.
2. Review the ontology contract you map in your Fabric tenant.
3. Build a Fabric Ontology Knowledge Source payload.
4. Build a combined Knowledge Base payload with Fabric and MCP sources.
5. Optionally retrieve with end-user source authorization.
6. Inspect offline Fabric and combined traces.

Primary manual: https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-fabric-ontology

## Runtime Safety

The notebook defaults to dry-run mode. Live Fabric retrieval requires tenant-specific values:

```text
RUN_LIVE_CALLS=true
SEARCH_ENDPOINT=https://<search-service>.search.windows.net
SEARCH_API_KEY=<search-admin-or-query-key>
FABRIC_WORKSPACE_ID=<fabric-workspace-guid>
FABRIC_ONTOLOGY_ID=<fabric-ontology-guid>
FABRIC_USER_SEARCH_TOKEN=<raw-user-token-for-https://search.azure.com/.default>
AZURE_OPENAI_ENDPOINT=https://<azure-openai-or-foundry-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT_ID=<chat-completions-deployment-name>
AZURE_OPENAI_MODEL_NAME=<model-name>
```

Pass `FABRIC_USER_SEARCH_TOKEN` as a raw token. Do not prefix it with `Bearer`.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
import urllib.error
import urllib.request
from collections import defaultdict
from pathlib import Path
from typing import Any

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from ks_factory import create_fabric_ontology_knowledge_source, create_knowledge_base
from ks_factory.diagnostics import summarize_activity, summarize_references


def load_env_file(path: str | None) -> None:
    if not path:
        return
    env_path = Path(path).expanduser()
    if not env_path.exists():
        raise FileNotFoundError(env_path)
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(os.getenv("LIVE_KS_ENV_FILE"))

SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT", "https://<search-service>.search.windows.net").rstrip("/")
SEARCH_API_VERSION = os.getenv("SEARCH_API_VERSION", "2026-05-01-preview")
SEARCH_API_KEY = os.getenv("SEARCH_API_KEY", "<search-admin-or-query-key-for-poc>")
RUN_LIVE_CALLS = os.getenv("RUN_LIVE_CALLS", "false").lower() == "true"

FABRIC_KS_NAME = os.getenv("FABRIC_ONTOLOGY_KNOWLEDGE_SOURCE_NAME", "fabric-ontology-ks")
FABRIC_WORKSPACE_ID = os.getenv("FABRIC_WORKSPACE_ID", "00000000-0000-0000-0000-000000000000")
FABRIC_ONTOLOGY_ID = os.getenv("FABRIC_ONTOLOGY_ID", "00000000-0000-0000-0000-000000000001")
FABRIC_USER_SEARCH_TOKEN = os.getenv("FABRIC_USER_SEARCH_TOKEN", "<raw-user-token-for-https://search.azure.com/.default>")

MCP_KS_NAME = os.getenv("MCP_KNOWLEDGE_SOURCE_NAME", "microsoft-learn-mcp-ks")
KNOWLEDGE_BASE_NAME = os.getenv("KNOWLEDGE_BASE_NAME", "live-knowledge-sources-kb")

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "https://<azure-openai-or-foundry-resource>.openai.azure.com")
AZURE_OPENAI_DEPLOYMENT_ID = os.getenv("AZURE_OPENAI_DEPLOYMENT_ID", "<chat-completions-deployment-name>")
AZURE_OPENAI_MODEL_NAME = os.getenv("AZURE_OPENAI_MODEL_NAME", "<model-name>")

print("RUN_LIVE_CALLS =", RUN_LIVE_CALLS)
print("Fabric KS =", FABRIC_KS_NAME)
print("Fabric workspace placeholder?", FABRIC_WORKSPACE_ID.startswith("00000000"))

## Inspect The Airline Ops Sample Data

The sample uses fictional carrier names and real airport geography. Semantic joins should use stable business fields such as `airline_code`, `delay_category`, `trigger_condition`, and `applicable_scope`, not real airline brand names.

In [ ]:
DATA_DIR = REPO_ROOT / "samples" / "data" / "airline-ops"


def read_csv(name: str) -> list[dict[str, str]]:
    with (DATA_DIR / name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


airlines = read_csv("airlines.csv")
airports = read_csv("airports.csv")
routes = read_csv("routes.csv")
flights = read_csv("flights.csv")
delay_events = read_csv("delay_events.csv")
policies = read_csv("passenger_care_policies.csv")
regulatory_refs = read_csv("regulatory_references.csv")

flight_to_airline = {row["flight_id"]: row["airline_code"] for row in flights}
airline_names = {row["airline_code"]: row["airline_name"] for row in airlines}
exposure_by_airline: dict[str, int] = defaultdict(int)
for event in delay_events:
    exposure_by_airline[flight_to_airline[event["flight_id"]]] += int(event["customer_care_exposure_usd"])

summary = {
    "airlines": len(airlines),
    "airports": len(airports),
    "routes": len(routes),
    "flights": len(flights),
    "delayed_flights_over_15_minutes": sum(int(row["arrival_delay_minutes"]) > 15 for row in flights),
    "delay_events": len(delay_events),
    "passenger_care_policies": len(policies),
    "regulatory_references": len(regulatory_refs),
    "customer_care_exposure_usd": sum(int(row["customer_care_exposure_usd"]) for row in delay_events),
    "controllable_delay_events": sum(row["controllable"].lower() == "true" for row in delay_events),
}

ranking = [
    {"airline_code": code, "airline_name": airline_names[code], "customer_care_exposure_usd": value}
    for code, value in sorted(exposure_by_airline.items(), key=lambda item: item[1], reverse=True)
]

print(json.dumps(summary, indent=2))
print(json.dumps(ranking, indent=2))

## Review The Ontology Contract

Map equivalent entities and relationships in your Fabric tenant before running live Fabric retrieval.

Contract path:

```text
samples/ontology/airline-ops/ontology-contract.yaml
```

In [ ]:
contract_path = REPO_ROOT / "samples" / "ontology" / "airline-ops" / "ontology-contract.yaml"
contract_preview = "\n".join(contract_path.read_text(encoding="utf-8").splitlines()[:80])
print(contract_preview)

## Build The Fabric Payloads

The Fabric payload binds Azure AI Search to an existing Fabric workspace and ontology item. The combined Knowledge Base includes both Fabric Ontology KS and Microsoft Learn MCP KS.

In [ ]:
fabric_knowledge_source = create_fabric_ontology_knowledge_source(
    name=FABRIC_KS_NAME,
    workspace_id=FABRIC_WORKSPACE_ID,
    ontology_id=FABRIC_ONTOLOGY_ID,
    description="Governed Airline Ops business-semantic grounding source from Microsoft Fabric.",
)

combined_knowledge_base = create_knowledge_base(
    name=KNOWLEDGE_BASE_NAME,
    knowledge_source_names=[MCP_KS_NAME, FABRIC_KS_NAME],
    description="Knowledge Base with Microsoft Learn MCP and Fabric Airline Ops ontology live Knowledge Sources.",
    retrieval_instructions=(
        "Use Fabric Ontology for Airline Ops entities, relationships, delay exposure, and policy matching. "
        "Use MCP Server for Microsoft Learn implementation guidance about Knowledge Sources and retrieve traces."
    ),
    azure_openai_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_openai_deployment_id=AZURE_OPENAI_DEPLOYMENT_ID,
    azure_openai_model_name=AZURE_OPENAI_MODEL_NAME,
)

print(json.dumps({"knowledgeSource": fabric_knowledge_source, "combinedKnowledgeBase": combined_knowledge_base}, indent=2))

## Optional Live Calls

This cell creates the Fabric Knowledge Source, creates/updates the combined Knowledge Base, and retrieves two questions:

- Fabric-only business question
- Combined business + implementation guidance question

It only executes when `RUN_LIVE_CALLS=true`.

In [ ]:
def search_url(path: str) -> str:
    if "<" in SEARCH_ENDPOINT:
        raise ValueError("Set SEARCH_ENDPOINT before running live calls.")
    return f"{SEARCH_ENDPOINT}{path}?api-version={SEARCH_API_VERSION}"


def request_json(method: str, path: str, body: dict[str, Any] | None = None, extra_headers: dict[str, str] | None = None) -> dict[str, Any]:
    headers = {
        "api-key": SEARCH_API_KEY,
        "Content-Type": "application/json",
        "Prefer": "return=representation",
    }
    if extra_headers:
        headers.update(extra_headers)
    data = None if body is None else json.dumps(body).encode("utf-8")
    request = urllib.request.Request(search_url(path), data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"{method} {path} failed: {error.code}\n{detail}") from error


fabric_retrieve_body = {
    "messages": [
        {"role": "user", "content": [{"type": "text", "text": "Which airlines have the highest customer-care exposure this month?"}]}
    ],
    "includeActivity": True,
    "knowledgeSourceParams": [
        {
            "kind": "fabricOntology",
            "knowledgeSourceName": FABRIC_KS_NAME,
            "includeReferences": True,
            "includeReferenceSourceData": True,
        }
    ],
    "outputMode": "answerSynthesis",
    "retrievalReasoningEffort": {"kind": "low"},
    "maxRuntimeInSeconds": 60,
}

combined_retrieve_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Using the Airline Ops ontology, identify the airline with the highest customer-care exposure this month. Also cite Microsoft Learn guidance for how I should validate activity, references, and sourceData in the Knowledge Base retrieve response.",
                }
            ],
        }
    ],
    "includeActivity": True,
    "knowledgeSourceParams": [
        {"kind": "fabricOntology", "knowledgeSourceName": FABRIC_KS_NAME, "includeReferences": True, "includeReferenceSourceData": True},
        {"kind": "mcpServer", "knowledgeSourceName": MCP_KS_NAME, "includeReferences": True, "includeReferenceSourceData": True},
    ],
    "outputMode": "answerSynthesis",
    "retrievalReasoningEffort": {"kind": "low"},
    "maxRuntimeInSeconds": 90,
}

if RUN_LIVE_CALLS:
    if FABRIC_USER_SEARCH_TOKEN.startswith("<"):
        raise ValueError("Set FABRIC_USER_SEARCH_TOKEN to a raw token for https://search.azure.com/.default")
    request_json("PUT", f"/knowledgesources/{FABRIC_KS_NAME}", fabric_knowledge_source)
    request_json("PUT", f"/knowledgebases/{KNOWLEDGE_BASE_NAME}", combined_knowledge_base)
    auth_headers = {"x-ms-query-source-authorization": FABRIC_USER_SEARCH_TOKEN}

    fabric_response = request_json("POST", f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/retrieve", fabric_retrieve_body, auth_headers)
    print("Fabric activity")
    print(json.dumps(summarize_activity(fabric_response), indent=2))
    print("Fabric references")
    print(json.dumps(summarize_references(fabric_response), indent=2))

    combined_response = request_json("POST", f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/retrieve", combined_retrieve_body, auth_headers)
    print("Combined activity")
    print(json.dumps(summarize_activity(combined_response), indent=2))
    print("Combined references")
    print(json.dumps(summarize_references(combined_response), indent=2))
else:
    print("Dry run. Set RUN_LIVE_CALLS=true to create resources and retrieve live Fabric results.")
    print(json.dumps({"fabricRetrieve": fabric_retrieve_body, "combinedRetrieve": combined_retrieve_body}, indent=2))

## Offline Replay

Use these checked-in responses to inspect the expected Fabric and combined trace shapes without live keys.

In [ ]:
for response_name in ["fabric-airline-ops-retrieve.sample.json", "combined-airline-ops-retrieve.sample.json"]:
    response_path = REPO_ROOT / "samples" / "responses" / response_name
    response = json.loads(response_path.read_text(encoding="utf-8"))
    print("\n===", response_name, "===")
    print("Activity")
    print(json.dumps(summarize_activity(response), indent=2))
    print("References")
    print(json.dumps(summarize_references(response), indent=2))

## Expected Result

A successful Fabric run should show:

- `activity[*].type == "fabricOntology"`
- Fabric references with `sourceData.fabricAnswer`
- Fabric references with `sourceData.fabricRawData`
- Airline Ops answers grounded in the ontology contract

For semantic join demos, keep carrier names fictional and join policy/regulation content through `delay_category`, `trigger_condition`, and scope fields.